In [ ]:
import pandas as pd
import re

In [ ]:
%%time
df = pd.read_csv("archive-cc-2022-nc.csv")
df

In [ ]:
%%time
df.head()

In [ ]:
%%time
df[df.archive_id.isin(['CNNW_20190929_090000_CNN_Newsroom_Live', 'MSNBCW_20200528_170000_MSNBC_Live'])]

In [ ]:
# Define a function for string replacement
def replace_strings(column):
    try:
        return column.apply(lambda x: x if pd.isna(x) else re.sub("b['\"](.*)['\"]", '\\1', x))
    except Exception as e:
        return column

In [ ]:
%%time
first = True
OUTPUT_FILE = '../shared/archive-cc-2022-all-nc.csv.gz'
for i, adf in enumerate(pd.read_csv('../shared/archive-cc-2022.csv.gz', chunksize=1000)):
    print(i)
    adf = adf.apply(lambda col: replace_strings(col), axis=0)
    idents = adf.identifier.tolist()
    cdf = df[df.archive_id.isin(idents)]
    #print(len(cdf))
    adf = pd.merge(adf, cdf, how='left', left_on='identifier', right_on='archive_id')
    del adf['archive_id']
    if first:
        adf.to_csv(OUTPUT_FILE, index=False, header=first, compression='gzip')
        first = False
    else:
        adf.to_csv(OUTPUT_FILE, mode='a', index=False, header=first, compression='gzip')
    #if i >= 10:
    #    break